# 推論

## 目的・方針

- `notebooks/model/build_model.ipynb` でUnity Catalog Model Registryに登録した日次売上数量予測モデルを使い、指定した1日について店舗×商品ごとの需要を推論するテンプレートnotebook
- モデル名・エイリアス・推論対象日は `dbutils.widgets` でパラメータ化し、別モデル・別日付でも同じnotebookを使い回せるようにする
- 特徴量は学習時と同じ列構成（store_id_code, product_id_code, category_code, unit_price, stock_quantity, day_of_week, is_weekend）を、`workspace.silver._20_silver_{products,inventory}` から再構築する。推論対象は1日分なので、学習時のようなカレンダー×店舗×商品のdenseパネル化（crossJoin）は不要で、店舗×商品マスタに指定日のカレンダー特徴を付与するだけでよい
- カテゴリ変数（store_id, product_id, category）は学習時と同じ全量マスタ（4店舗・16商品・5カテゴリ）から `.astype("category").cat.codes` で導出するため、学習時と同じコード体系になる想定（対象ドメインが変わらない前提。本番運用ではエンコーダの永続化が望ましいが、practiceのシンプルなテンプレートとしてこの前提を許容する）
- 推論結果は `display()` で確認する（出力テーブルへの書き込みはスコープ外とし、シンプルに保つ）

In [ ]:
dbutils.widgets.text("model_name", "workspace.model.daily_sales_quantity_predictor")
dbutils.widgets.text("model_alias", "champion")
dbutils.widgets.text("inference_date", "")  # 空欄なら当日日付を使用

In [ ]:
from pyspark.sql.functions import *
import pandas as pd
from datetime import date

import mlflow

mlflow.set_registry_uri("databricks-uc")

In [ ]:
model_name = dbutils.widgets.get("model_name")
model_alias = dbutils.widgets.get("model_alias")
inference_date_str = dbutils.widgets.get("inference_date")

inference_date = date.fromisoformat(inference_date_str) if inference_date_str else date.today()
print(f"model: {model_name}@{model_alias}, inference_date: {inference_date}")

In [ ]:
model = mlflow.pyfunc.load_model(f"models:/{model_name}@{model_alias}")

In [ ]:
products_df = spark.read.table("workspace.silver._20_silver_products")
inventory_df = spark.read.table("workspace.silver._20_silver_inventory")

score_df = (
    inventory_df.select("store_id", "product_id", "stock_quantity")
    .join(products_df.select("product_id", "category", "unit_price"), on="product_id", how="left")
    .withColumn("sale_date", lit(inference_date))
    .withColumn("day_of_week", dayofweek("sale_date"))
    .withColumn("is_weekend", when(col("day_of_week").isin(1, 7), 1).otherwise(0))
)

In [ ]:
pdf = score_df.toPandas()  # 店舗4×商品16=64行程度でサイズ的に安全

for c in ["store_id", "product_id", "category"]:
    pdf[f"{c}_code"] = pdf[c].astype("category").cat.codes

FEATURE_COLS = [
    "store_id_code",
    "product_id_code",
    "category_code",
    "unit_price",
    "stock_quantity",
    "day_of_week",
    "is_weekend",
]

In [ ]:
pdf["predicted_quantity"] = model.predict(pdf[FEATURE_COLS])

result_df = pdf[["sale_date", "store_id", "product_id", "predicted_quantity"]]
display(result_df)